In [1]:
# Importing packages
import os
import csv
import json
import pandas as pd
import numpy as np
from pathlib import Path

In [3]:
# Defining paths for data retrieval and storage
table_file_name = 'manifest.csv'
ds_folder = 'dataset_final'
page_folders = ['Home Page', 'News Listing', 'Article Pages']
news_sites_list = os.listdir(ds_folder)

In [4]:
# Iterate over the news_sites_list and populate the imgs_list_full list with the images found in the dataset folder
imgs_list_full = []

for site in news_sites_list:
    for page_folder in page_folders:
        # Check if the current page folder is the article pages one
        if page_folder == 'Article Pages':  # If yes, then access the images in the 'imgs' subdirectory
            full_path = os.path.join(ds_folder, site, page_folder, 'imgs')
        else:   # If no, access the images as normal
            full_path = os.path.join(ds_folder, site, page_folder)
        imgs_list = [os.path.join(full_path, f) for f in os.listdir(full_path) if f.lower().endswith('.png') or f.endswith('.jpg') or f.endswith('.jpeg')]
        imgs_list_full.append(imgs_list)

# Printing the output for testing purposes
imgs_list_full

[['dataset_final\\Independent\\Home Page\\0001.png',
  'dataset_final\\Independent\\Home Page\\0002.png',
  'dataset_final\\Independent\\Home Page\\0003.png',
  'dataset_final\\Independent\\Home Page\\0004.png',
  'dataset_final\\Independent\\Home Page\\0005.png',
  'dataset_final\\Independent\\Home Page\\0006.png',
  'dataset_final\\Independent\\Home Page\\0007.png',
  'dataset_final\\Independent\\Home Page\\0008.png',
  'dataset_final\\Independent\\Home Page\\0009.png',
  'dataset_final\\Independent\\Home Page\\0010.png',
  'dataset_final\\Independent\\Home Page\\0011.png',
  'dataset_final\\Independent\\Home Page\\0012.png',
  'dataset_final\\Independent\\Home Page\\0013.png',
  'dataset_final\\Independent\\Home Page\\0014.png',
  'dataset_final\\Independent\\Home Page\\0015.png',
  'dataset_final\\Independent\\Home Page\\0016.png',
  'dataset_final\\Independent\\Home Page\\0017.png',
  'dataset_final\\Independent\\Home Page\\0018.png',
  'dataset_final\\Independent\\Home Page\\0019

In [5]:
homepage_urls_list = {
    'Independent': 'https://www.independent.com.mt/',
    'Malta Daily': 'https://maltadaily.mt/',
    'Malta Today': 'https://www.maltatoday.com.mt/',
    'Newsbook': 'https://newsbook.com.mt/',
    'The Shift': 'https://theshiftnews.com/',
    'Times of Malta': 'https://timesofmalta.com/',
}

In [6]:
listing_urls_list_file = 'listing_urls_list.json'
article_pages_urls_list_file = 'article_urls_list.json'

# Only for Newsbook
article_pages_en_urls_list_file = 'article_urls_list_en.json'
article_pages_mt_urls_list_file = 'article_urls_list_mt.json'

In [7]:
def load_url_data(path):
    if not os.path.exists(path):
        return None

    with open(path, 'r') as f:
        data = json.load(f)
        
    return data

def get_listing_img_url(url_data, img_name):
    img_id = os.path.splitext(img_name)[0]
    url = url_data.get("listing_urls", {}).get(img_id, np.nan)
    return url

def get_article_img_url(url_data, img_name):
    img_id = os.path.splitext(img_name)[0]
    url = url_data.get("article_urls", {}).get(img_id, np.nan)
    return url

In [9]:
d = {
    'id': [],
    'website': [],
    'page_type': [],
    'screenshot_path': [],
    'html_path': [],
    'url': [],
}

for imgs_list in imgs_list_full:
    for img_path in imgs_list:
        img_path_parts = img_path.split('\\')
        id = os.path.splitext(img_path_parts[-1])[0]
        page_type = str(img_path_parts[2]).lower().replace(' ', '_')
        website = str(img_path_parts[1]).lower().replace(' ', '_')
        html_path_file = os.path.join('data', 'html', website, 'html_paths.json')

        d['id'].append(id)
        d['page_type'].append(page_type)
        d['website'].append(website)
        d['screenshot_path'].append(img_path)

        if not os.path.exists(html_path_file):
            d['html_path'].append(np.nan)
        else:
            with open(html_path_file, 'r') as f:
                html_path = json.load(f)
            print(html_path)

            html_path_id = html_path['html_paths'][id]
            d['html_path'].append(os.path.join('data', 'html', website, html_path_id))
        if img_path_parts[2] == 'Home Page':
            d['url'].append(homepage_urls_list.get(img_path_parts[1]))
        elif img_path_parts[2] == 'News Listing':
            print(os.path.join(img_path_parts[0], img_path_parts[1], img_path_parts[2], listing_urls_list_file))
            url_data = load_url_data(os.path.join(img_path_parts[0], img_path_parts[1], img_path_parts[2], listing_urls_list_file))
            if url_data is None:
                d['url'].append(np.nan)
            else:
                img_url = get_listing_img_url(url_data, os.path.basename(img_path))
                print(img_url)
                d['url'].append(img_url)
        elif img_path_parts[2] == 'Article Pages':
            print(os.path.join(img_path_parts[0], img_path_parts[1], img_path_parts[2], 'urls', article_pages_urls_list_file))
            print(os.path.exists(os.path.join(img_path_parts[0], img_path_parts[1], img_path_parts[2], 'urls', article_pages_urls_list_file)))
            url_data = load_url_data(os.path.join(img_path_parts[0], img_path_parts[1], img_path_parts[2], 'urls', article_pages_urls_list_file))
            if url_data is None:
                d['url'].append(np.nan)
            else:
                img_url = get_article_img_url(url_data, os.path.basename(img_path))
                print(img_url)
                d['url'].append(img_url)
        else:
            d['url'].append(np.nan)
        # print('News Website:',img_path_parts[1])
        # print('Page Type:',img_path_parts[2])
        # print('Image ID:',img_path_parts[3])
        # print('--------------------------')
    # print(imgs_list)

{'html_paths': {'0001': 'home_page_fedb7f801a.html', '0002': 'home_page_fedb7f801a.html', '0003': 'home_page_fedb7f801a.html', '0004': 'home_page_fedb7f801a.html', '0005': 'home_page_fedb7f801a.html', '0006': 'home_page_fedb7f801a.html', '0007': 'home_page_fedb7f801a.html', '0008': 'home_page_fedb7f801a.html', '0009': 'home_page_fedb7f801a.html', '0010': 'home_page_fedb7f801a.html', '0011': 'home_page_fedb7f801a.html', '0012': 'home_page_fedb7f801a.html', '0013': 'home_page_fedb7f801a.html', '0014': 'home_page_fedb7f801a.html', '0015': 'home_page_fedb7f801a.html', '0016': 'home_page_fedb7f801a.html', '0017': 'home_page_fedb7f801a.html', '0018': 'home_page_fedb7f801a.html', '0019': 'home_page_fedb7f801a.html', '0020': 'home_page_fedb7f801a.html', '0021': 'home_page_fedb7f801a.html', '0022': 'home_page_fedb7f801a.html', '0023': 'home_page_fedb7f801a.html', '0024': 'home_page_fedb7f801a.html', '0025': 'home_page_fedb7f801a.html', '0026': 'home_page_fedb7f801a.html', '0027': 'home_page_fed

In [10]:
print(len(d['id']))
print(len(d['page_type']))
print(len(d['screenshot_path']))
print(len(d['website']))
print(len(d['url']))

982
982
982
982
982


In [11]:
df = pd.DataFrame(d)

df.head()

,id,website,page_type,screenshot_path,html_path,url
0,0001,independent,home_page,dataset_final\Independent\Home Page\0001.png,data\html\independent\home_page_fedb7f801a.html,https://www.independent.com.mt/
1,0002,independent,home_page,dataset_final\Independent\Home Page\0002.png,data\html\independent\home_page_fedb7f801a.html,https://www.independent.com.mt/
2,0003,independent,home_page,dataset_final\Independent\Home Page\0003.png,data\html\independent\home_page_fedb7f801a.html,https://www.independent.com.mt/
3,0004,independent,home_page,dataset_final\Independent\Home Page\0004.png,data\html\independent\home_page_fedb7f801a.html,https://www.independent.com.mt/
4,0005,independent,home_page,dataset_final\Independent\Home Page\0005.png,data\html\independent\home_page_fedb7f801a.html,https://www.independent.com.mt/


In [ ]:
# df.to_csv(os.path.join(ds_folder, table_file_name))
df.to_csv(table_file_name, index=False)